In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

In [2]:
# Load the MNIST training dataset
training_data = datasets.MNIST(
    root="data",                     # Directory where data will be stored
    train=True,                      # Load the training set
    download=True,                   # Download the dataset if not already present
    transform=transforms.ToTensor()  # Convert images to PyTorch tensors (from PIL images)
)

# Correction: `train=False` to load the test set
test_data = datasets.MNIST(
    root="data",                     # Same directory
    train=False,                     # Load the test set instead of the training set
    download=True,                   # Download if not already present
    transform=transforms.ToTensor()  # Convert images to tensors
)


100%|██████████| 9.91M/9.91M [00:00<00:00, 53.6MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.67MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 14.4MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 4.83MB/s]


In [3]:
# Set the number of samples to be processed in each batch
batch_size = 64

# Wrap the training dataset in a DataLoader to enable batch processing
train_dataloader = DataLoader(training_data, batch_size=batch_size)

# Wrap the test dataset in a DataLoader
test_dataloader = DataLoader(test_data, batch_size=batch_size)

# Automatically use GPU (CUDA) if available, otherwise fall back to CPU
device = "cuda" if torch.cuda.is_available() else "cpu"

# Print which device is being used
print(f"Using {device} device")  # Note: added space after "Using" for correct formatting


Using cpu device


In [4]:
# Define a neural network model by subclassing nn.Module
class Net(nn.Module):
    def __init__(self):
        super().__init__()  # Initialize the parent nn.Module class

        # Layer to flatten 2D input image (28x28) into 1D vector (784)
        self.flatten = nn.Flatten()

        # A stack of layers applied sequentially
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),  # First fully connected layer: 784 input features → 512 output features
            nn.ReLU(),              # ReLU activation adds non-linearity
            nn.Linear(512, 512),    # Second fully connected layer: 512 → 512
            nn.ReLU(),              # ReLU activation
            nn.Linear(512, 10)      # Output layer: 512 → 10 classes (digits 0–9)
        )

    # Defines the forward pass of the model
    def forward(self, x):
        x = self.flatten(x)                 # Flatten the 28x28 image into a vector
        logits = self.linear_relu_stack(x)  # Pass through the stacked layers
        return logits                       # Return unnormalized class scores (logits)

# Instantiate the model and move it to the appropriate device (GPU or CPU)
model = Net().to(device)

# Print the model architecture
print(model)


Net(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [5]:
# Define the loss function for multi-class classification
# CrossEntropyLoss combines LogSoftmax and Negative Log Likelihood internally
loss_fn = nn.CrossEntropyLoss()

# Define the optimizer: Adam adapts learning rates for each parameter
# model.parameters() returns all trainable parameters of the model
# lr=1e-3 sets the learning rate to 0.001
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Function to train the model for one epoch
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)     # Total number of samples
    model.train()                      # Set model to training mode (activates dropout/batchnorm if used)

    for batch, (X, y) in enumerate(dataloader):
        # Move data and labels to the specified device (CPU or GPU)
        X, y = X.to(device), y.to(device)

        # Forward pass: compute predictions
        pred = model(X)
        # Compute the loss
        loss = loss_fn(pred, y)

        # Backward pass: compute gradients
        loss.backward()
        # Update weights using optimizer
        optimizer.step()
        # Reset gradients to zero for the next step
        optimizer.zero_grad()

        # Print loss every 100 batches to monitor progress
        if batch % 100 == 0:
            loss_val, current = loss.item(), batch * len(X)
            print(f"loss: {loss_val:>7f}  [{current:>5d}/{size:>5d}]")

# Function to evaluate model performance on test data
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)     # Total number of test samples
    num_batches = len(dataloader)      # Number of batches in test DataLoader
    model.eval()                       # Set model to evaluation mode (disables dropout, etc.)
    test_loss, correct = 0, 0

    # Disable gradient calculation for inference
    with torch.inference_mode():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)

            # Forward pass
            pred = model(X)
            # Accumulate loss
            test_loss += loss_fn(pred, y).item()
            # Count correct predictions
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    # Compute average loss and accuracy
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%  Avg loss: {test_loss:>8f} \n")


In [6]:
epochs = 5

for t in range(epochs):
  print(f"Epoch {t+1}\n---------------")
  train(train_dataloader, model,loss_fn, optimizer)
  test(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
---------------
loss: 2.304720  [    0/60000]
loss: 0.258314  [ 6400/60000]
loss: 0.191814  [12800/60000]
loss: 0.225043  [19200/60000]
loss: 0.169728  [25600/60000]
loss: 0.297681  [32000/60000]
loss: 0.120546  [38400/60000]
loss: 0.224069  [44800/60000]
loss: 0.288540  [51200/60000]
loss: 0.174027  [57600/60000]
Test Error: 
 Accuracy: 95.5%  Avg loss: 0.138295 

Epoch 2
---------------
loss: 0.098606  [    0/60000]
loss: 0.095633  [ 6400/60000]
loss: 0.088632  [12800/60000]
loss: 0.095351  [19200/60000]
loss: 0.042947  [25600/60000]
loss: 0.128627  [32000/60000]
loss: 0.048396  [38400/60000]
loss: 0.116585  [44800/60000]
loss: 0.119611  [51200/60000]
loss: 0.118684  [57600/60000]
Test Error: 
 Accuracy: 96.8%  Avg loss: 0.103971 

Epoch 3
---------------
loss: 0.075001  [    0/60000]
loss: 0.047093  [ 6400/60000]
loss: 0.059596  [12800/60000]
loss: 0.082185  [19200/60000]
loss: 0.033189  [25600/60000]
loss: 0.070615  [32000/60000]
loss: 0.040851  [38400/60000]
loss: 0.065253

In [7]:
torch.save(model.state_dict(),"mnist_base_model.pth")
print("Saved pyTorch Model State to mnist_base_model.pth")

Saved pyTorch Model State to mnist_base_model.pth


In [10]:
model = Net().to(device)
model.load_state_dict(torch.load("mnist_base_model.pth"))
model.eval()


from PIL import Image

image_path ="5.jpg"
image = Image.open(image_path).convert("L")
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((28,28)),
])
image = transform(image).to(device)

with torch.inference_mode():
  output = model(image)
  prediction = torch.argmax(output).item()
print(f"predicition: {prediction}")

predicition: 5


In [14]:
# Move the model to the specified device (CPU or GPU)
model = Net().to(device)

# Load the pre-trained model weights from file
model.load_state_dict(torch.load("mnist_base_model.pth"))

# Set the model to evaluation mode (important for inference, disables dropout/batchnorm updates)
model.eval()

from PIL import Image
from torchvision import transforms  # Make sure to import this

# Load and convert the image to grayscale (L mode)
image_path = "5.jpg"
image = Image.open(image_path).convert("L")

# Define the transformation: resize and convert to tensor
transform = transforms.Compose([
    transforms.ToTensor(),              # Convert image to PyTorch tensor and normalize to [0, 1]
    transforms.Resize((28, 28)),        # Resize the image to 28x28 (MNIST input size)
])

# Apply the transformation and move the image tensor to the same device as the model
image = transform(image).to(device)

# Disable gradient calculation (more efficient during inference)
with torch.inference_mode():
    # Forward pass: get model output
    output = model(image)  # Add batch dimension (1, 1, 28, 28)

    # Get the predicted class by finding the index of the max logit
    prediction = torch.argmax(output).item()

# Print the prediction
print(f"prediction: {prediction}")


prediction: 5
